# 04 · Train your first neural image classifier

**Level:** early intermediate · **Time:** 45–65 minutes · **Data:** bundled digits · **Hardware:** local GPU or CPU.

Build a multilayer perceptron (MLP), follow an explicit training loop, inspect learning curves,
and save a reproducible checkpoint. An MLP connects flattened pixels through learned linear
layers and a nonlinear activation. **Prerequisite:** tensors and autograd from notebook 03.
This notebook independently loads all data and trains its own model.

Run the default experiment once before modifying it. Normal mode uses eight epochs;
`COURSE_SMOKE_TEST=1` uses one epoch and much less data, so its scores are not learning results.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


## Step 1 — Load separate datasets in mini-batches
A `DataLoader` groups examples into batches. Training shuffles their order; validation and
test use a fixed order. Our shared loader uses `num_workers=0` for a simple Windows setup.
Each image is `(1, 8, 8)` and each label is a scalar integer. This dataset needs no download.


In [ ]:
import copy
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay
from course_utils import get_digits_loaders

seed_everything(42)
train_loader, val_loader, test_loader = get_digits_loaders(batch_size=64)
sample_images, sample_labels = next(iter(train_loader))
print("Batch:", tuple(sample_images.shape), "labels:", tuple(sample_labels.shape))
print("Examples:", {name: len(loader.dataset) for name, loader in
                    [("train", train_loader), ("validation", val_loader), ("test", test_loader)]})
print("Dtypes:", sample_images.dtype, sample_labels.dtype)


## Step 2 — Build a small network and count its parameters
`Flatten` converts each image into 64 inputs. `Linear(64, 64)` learns weighted combinations,
`ReLU` keeps positive activations and replaces negative ones with zero, and `Linear(64, 10)`
produces one raw score (a **logit**) per class. The nonlinearity lets stacked layers represent
more than one linear mapping.


In [ ]:
class DigitMLP(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(), nn.Linear(8 * 8, hidden_size),
            nn.ReLU(), nn.Linear(hidden_size, 10))

    def forward(self, images):
        return self.network(images)

model = DigitMLP(hidden_size=64).to(device)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Expected: (64*64 + 64) + (64*10 + 10) =", 64*64 + 64 + 64*10 + 10)


## Step 3 — Keep logits for the loss; use softmax for inspection
`CrossEntropyLoss` expects raw logits of shape `(batch, 10)` and integer targets of shape
`(batch,)`. It performs the required normalization internally. **Do not add softmax to the
model before this loss.** For display, softmax converts logits into probabilities that sum to 1.
`argmax` picks the same class from either logits or softmax values.


In [ ]:
criterion = nn.CrossEntropyLoss()
with torch.no_grad():
    sample_logits = model(sample_images.to(device))
    sample_probabilities = sample_logits.softmax(dim=1)
    starting_loss = criterion(sample_logits, sample_labels.to(device)).item()
print("Logits shape:", tuple(sample_logits.shape))
print("Probability sums:", sample_probabilities[:4].sum(dim=1).cpu().numpy())
print("Untrained batch loss:", round(starting_loss, 4), "| log(10):", round(np.log(10), 4))
print("Same argmax:", torch.equal(sample_logits.argmax(1), sample_probabilities.argmax(1)))


## Step 4 — Declare the experiment before fitting
Adam adjusts step sizes for individual parameters. One **epoch** visits the training dataset once;
one **step** updates weights from one mini-batch. We select the checkpoint with the smallest
validation loss over the declared eight epochs. Test data plays no part in this decision.


In [ ]:
epochs = 1 if SMOKE_TEST else 8
learning_rate = 0.003
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
history = []
best_val_loss = float("inf")
best_state = None
best_epoch = None
print("Epochs:", epochs, "| training steps per epoch:", len(train_loader))


## Step 5 — Read and run the complete training loop
Follow the numbered statements: move the batch → clear old gradients → forward pass →
calculate loss → backward pass → update weights. `model.train()` enables training behavior.
Validation uses `model.eval()` and `torch.inference_mode()` to avoid gradient recording.

We multiply each batch's mean loss by its batch size before averaging, so a smaller final
batch receives the correct weight. Training metrics average batches while parameters are
changing; validation measures the model at the end of the epoch.


In [ ]:
for epoch in range(1, epochs + 1):
    model.train()
    train_loss_sum, train_correct, train_count = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)  # 1. Device
        optimizer.zero_grad(set_to_none=True)                # 2. Clear
        logits = model(images)                              # 3. Forward
        loss = criterion(logits, labels)                     # 4. Loss
        loss.backward()                                      # 5. Backward
        optimizer.step()                                     # 6. Update

        batch_size = labels.size(0)
        train_loss_sum += loss.item() * batch_size
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_count += batch_size

    model.eval()
    val_loss_sum, val_correct, val_count = 0.0, 0, 0
    with torch.inference_mode():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            batch_size = labels.size(0)
            val_loss_sum += loss.item() * batch_size
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_count += batch_size

    row = {"epoch": epoch, "train_loss": train_loss_sum / train_count,
           "val_loss": val_loss_sum / val_count,
           "train_accuracy": train_correct / train_count,
           "val_accuracy": val_correct / val_count}
    history.append(row)
    if row["val_loss"] < best_val_loss:
        best_val_loss = row["val_loss"]
        best_epoch = epoch
        # A deep copy preserves the selected weights while later epochs keep training.
        best_state = copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch:02d}: train loss={row['train_loss']:.4f}, "
          f"val loss={row['val_loss']:.4f}, val accuracy={row['val_accuracy']:.3f}")

model.load_state_dict(best_state)
model.eval()
print("Restored validation-selected epoch:", best_epoch)


## Step 6 — Read learning curves
Expect loss to fall and accuracy to rise in normal mode. If training improves while
validation deteriorates, the model may be overfitting. Both staying poor can indicate an
optimization problem, unsuitable representation or a data bug. These are clues, not guarantees.
A single smoke-test epoch verifies execution; it cannot reveal a meaningful curve.


In [ ]:
history_table = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name in ["train", "val"]:
    axes[0].plot(history_table["epoch"], history_table[f"{name}_loss"], marker="o", label=name)
    axes[1].plot(history_table["epoch"], history_table[f"{name}_accuracy"], marker="o", label=name)
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy loss")
axes[1].set(xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1.05))
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()
history_table.round(4)


## Step 7 — Freeze the choice and evaluate once on held-out test data
Collect logits, labels and images in one pass. The following metrics and plots all reuse
those predictions. Macro F1 gives each class equal weight. Do not change training settings
because of this score; choose experiment changes using validation data.


In [ ]:
all_logits, all_targets, all_images = [], [], []
with torch.inference_mode():
    for images, labels in test_loader:
        all_logits.append(model(images.to(device)).cpu())
        all_targets.append(labels)
        all_images.append(images)
test_logits = torch.cat(all_logits)
test_targets = torch.cat(all_targets).numpy()
test_images = torch.cat(all_images).numpy()
test_predictions = test_logits.argmax(1).numpy()
test_probabilities = test_logits.softmax(1).numpy()
test_metrics = {
    "accuracy": float(accuracy_score(test_targets, test_predictions)),
    "macro_f1": float(f1_score(test_targets, test_predictions, average="macro", zero_division=0))}
print("Final test metrics:", test_metrics)
ConfusionMatrixDisplay.from_predictions(
    test_targets, test_predictions, labels=np.arange(10), cmap="Blues", colorbar=False)
plt.title("Final test confusion matrix")
plt.show()


## Step 8 — Inspect predictions and save the experiment
A high softmax value is the model's confidence, not proof of correctness. We will later
study calibration. Saving a `state_dict` stores learned parameter tensors; architecture and
preprocessing must also be known when loading. Save portable CPU weights plus useful metadata.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 4))
for index, ax in enumerate(axes.flat):
    prediction = test_predictions[index]
    confidence = test_probabilities[index, prediction]
    ax.imshow(test_images[index, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"True {test_targets[index]}, pred {prediction}\np={confidence:.2f}",
                 color="green" if prediction == test_targets[index] else "red")
    ax.axis("off")
plt.tight_layout()
plt.show()

output_dir = ARTIFACTS / "04_mlp"
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / "digit_mlp.pt"
torch.save({"model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},
            "hidden_size": 64, "classes": list(range(10)), "input_shape": [1, 8, 8],
            "preprocessing": "sklearn digits pixel values divided by 16",
            "selected_epoch": best_epoch, "seed": 42,
            "smoke_test": SMOKE_TEST}, checkpoint_path)
history_table.to_csv(output_dir / "history.csv", index=False)
(output_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2), encoding="utf-8")
print("Saved:", checkpoint_path)


## Practice 1 — Verify that saving and loading preserves predictions
Instantiate a fresh `DigitMLP`, load the saved state and compare its logits to the trained
model on a validation batch. Use `weights_only=True` for this tensor-and-primitive checkpoint.
Only load checkpoint files you trust.

<details><summary>Sample answer</summary>

```python
saved = torch.load(checkpoint_path, map_location=device, weights_only=True)
restored = DigitMLP(saved["hidden_size"]).to(device)
restored.load_state_dict(saved["model_state"])
restored.eval()
probe_images, _ = next(iter(val_loader))
with torch.inference_mode():
    print(torch.allclose(model(probe_images.to(device)), restored(probe_images.to(device))))
```

Identical architecture, weights, mode and preprocessing should give matching predictions.
</details>


In [ ]:
saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
print("Checkpoint architecture:", saved["input_shape"], "hidden units:", saved["hidden_size"])
# Create a fresh model, load its state, and compare validation logits.


## Practice 2 — Explain one manual cross-entropy calculation
Take the first validation example, convert logits to log probabilities with `log_softmax`,
and negate the log probability assigned to its true class. Compare with `criterion` on that example.

<details><summary>Sample answer</summary>

```python
images, labels = next(iter(val_loader))
with torch.inference_mode():
    logits = model(images[:1].to(device))
    manual_loss = -logits.log_softmax(1)[0, int(labels[0])]
    library_loss = criterion(logits, labels[:1].to(device))
print(manual_loss.item(), library_loss.item())
```

Assigning a higher probability to the correct class lowers this loss.
</details>


In [ ]:
probe_images, probe_labels = next(iter(val_loader))
print("True class of the probe:", int(probe_labels[0]))
# Compare manual and library cross entropy below.


## Practice 3 — Plan a controlled architecture comparison
Count parameters for hidden widths 16, 64 and 128. Predict the effect on capacity and
overfitting. As a further experiment, restart from the same seed, train each width using
the same split and learning rate, and compare **validation** loss. Skip the test cell while tuning.

<details><summary>Sample answer</summary>

```python
for width in [16, 64, 128]:
    trial_model = DigitMLP(hidden_size=width)
    print(width, sum(p.numel() for p in trial_model.parameters()))
```

Counts are 1,210; 4,810; and 9,610. More parameters increase capacity, but do not guarantee
better held-out performance. Record validation results before selecting a model.
</details>


In [ ]:
widths_to_compare = [16, 64, 128]
print("Declared comparison:", widths_to_compare)
# Count parameters now; use a fresh kernel for each later training trial.


## What you learned
You built a neural network, distinguished logits from probabilities, trained in mini-batches,
restored the validation-selected checkpoint, examined held-out predictions and saved artifacts.
You can now read the core code behind most supervised image-classification training loops.

**Checkpoint:** explain why validation has no `backward()` or `optimizer.step()`, and why
`model.eval()` and disabling gradient recording solve different problems.

**Next:** [05 · Convolutional neural networks](05_convolutional_neural_networks.ipynb).

Primary references: [building a model](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html),
[training loops](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html),
[cross entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html),
[saving and loading](https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html).
